<a href="https://colab.research.google.com/github/lapshinaaa/recsys-tasks/blob/main/DeepRecSys3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep RecSys Course
## Neural Ranking №3

In this task, we'll explore the methods of encoding categorical and numerical features, and then train a multi-task neural ranking model based on DCN V2 with mixture of experts.

Our model will be solving two problems in parallel:
- prediction of listening to a track;
- prediction of a like to a track.

# 1. Dataset Preparation

## yambda-50m-lag-features

We will use a dataset with precomputed Yambda features.

How this dataset was constructed:

1. The Yambda dataset `flat/50m multi_event` was taken as the base.
2. Only non-organic events were retained, that is, events obtained from recommendation surfaces, for example from My Wave in Yandex Music.
3. `is_skip = True` if the user listened to less than half of the track.
4. `is_full_play = True` if the user listened to more than 95% of the track.
5. In Yambda, a like and a listen are different events. At the same time, the model is trained on listening events: features are computed specifically for them, and the model must predict targets specifically for them. Therefore, likes must be attributed to listens, that is, we need to determine which exact listen each like belongs to. For this, each like is attached to the nearest-in-time listen of the same `(uid, item_id)` within 24 hours. If there are two suitable listens, the nearest one is chosen; in case of a tie, the previous one is chosen. After that, `is_like = True` for a listen if at least one like was attached to it.
6. After that, real-valued counters are computed over the dataset. They are divided into three types: user counters, track counters, and cross-counters. Importantly, for each listen with `timestamp`, all counters are computed not at the moment of the event itself, but at the moment `timestamp - lag_seconds`, where `lag_seconds = 15 minutes`. This delay is needed to prevent leakage during training and evaluation. The idea is that when making a prediction for the current listen, the model must not use information that appeared only after this event or too close to it and would not yet have had time to reach the features in a real system. This is especially important in our setup because later quality will be evaluated on temporally adjacent listens. This means that when comparing the current track with the previous one, we must not allow the features of the current object to already contain feedback on the previous track that should not yet have been available at prediction time. Otherwise, the model would effectively be peeking at the answer, and quality would be overestimated. In addition, such a lag better matches how features are usually updated in real recommendation systems: not instantly, but with some delay.
7. `is_like` and `is_full_play` are used as targets in the multi-task learning setup.

The dataset construction code can be viewed here: https://huggingface.co/datasets/matfu21/yambda-50m-lag-features/blob/main/event_processor.py#L296

In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import polars as pl
import tests
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="matfu21/yambda-50m-lag-features",
    repo_type="dataset",
    filename="listens.parquet",
)
listens = pl.read_parquet(path)

listens.parquet:   0%|          | 0.00/742M [00:00<?, ?B/s]

## Keep only preference-bearing pairs

In ranking tasks, the model is usually trained not on all events indiscriminately, but on specially selected objects that carry a signal about user preferences. This is especially important for relatively small ranking models. If such a model is trained on all examples in sequence, it will tend to learn global probabilities of like or full play more than it learns to correctly order temporally close objects. As a result, quality on the local pairwise comparisons that we actually care about may deteriorate. Therefore, in practice, training datasets for ranking are often sampled so as to retain only examples with useful feedback.

At the same time, it is important to understand that this is more of an engineering compromise than a fundamental limitation. If we had very large models with high expressive power and no strict computational constraints, then ideally we really would want to learn from all available events. Such a model could simultaneously recover global patterns well and also model local preferences with high quality. But in real recommendation systems, ranking models usually remain relatively compact, so it is useful to help them focus specifically on the examples that matter most for ranking.

Why ranking models are usually relatively small: such models operate at the final ranking stage, where it is necessary to recompute scores very quickly for a large number of candidates. The system typically has strict constraints on latency, memory, and queries per second. Therefore, it is difficult to use overly large models here: they would be too expensive and too slow in production. Precisely because of this, it is especially important that a small or medium-sized model spend its expressive capacity on genuinely useful examples rather than on predicting averaged global probabilities.

In this assignment, you need to keep only those listening events that belong to at least one local pair with differing feedback. The idea is as follows: if the current track and the temporally adjacent track have different signals, then such a pair helps train ranking. But if both on the left and on the right the feedback is the same, then the current event is less useful for this setup.

### What needs to be done

For each event, you need to:
1. look at the previous event of the same user;
2. look at the next event of the same user;
3. check whether `is_like` or `is_full_play` differs on at least one side;
4. keep only those rows for which such a difference exists.

### Important detail about sorting

Before looking at neighboring events, the data must be sorted by:
- `uid`,
- `timestamp`,
- `row_id`.

The `row_id` field is needed for determinism. The same user may have events with identical `timestamp`, and in that case sorting only by `uid` and `timestamp` does not define a unique order. Adding `row_id` makes that order fixed and reproducible.

In [6]:
def extract_preference_pairs(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .with_row_index("row_id")
        .sort(["uid", "timestamp", "row_id"])
        .with_columns([
            pl.col("is_like").shift(1).over("uid").alias("prev_is_like"),
            pl.col("is_like").shift(-1).over("uid").alias("next_is_like"),
            pl.col("is_full_play").shift(1).over("uid").alias("prev_is_full_play"),
            pl.col("is_full_play").shift(-1).over("uid").alias("next_is_full_play"),
        ])
        .with_columns([
            (
                ((pl.col("is_like") != pl.col("prev_is_like")) |
                 (pl.col("is_full_play") != pl.col("prev_is_full_play")))
                & pl.col("prev_is_like").is_not_null()
            ).alias("diff_prev"),
            (
                ((pl.col("is_like") != pl.col("next_is_like")) |
                 (pl.col("is_full_play") != pl.col("next_is_full_play")))
                & pl.col("next_is_like").is_not_null()
            ).alias("diff_next"),
        ])
        .filter(pl.col("diff_prev") | pl.col("diff_next"))
        .drop([
            "prev_is_like",
            "next_is_like",
            "prev_is_full_play",
            "next_is_full_play",
            "diff_prev",
            "diff_next",
            "row_id",
        ])
    )

listens = extract_preference_pairs(listens)
tests.test_extract_preference_pairs(listens)

All good! :)


In [7]:
listens.shape

(6885472, 23)

## The Yambda dataset

In this assignment, we will use the Yambda dataset: https://huggingface.co/datasets/yandex/yambda. From it, we will obtain information about the albums to which a track belongs, as well as about the artists of that track. These data will be needed to construct multivalent features.

Below is the class from Hugging Face that you will need to use.

In [8]:
from dataclasses import dataclass


@dataclass
class DatasetConfig:
    dataset_type: str = 'flat'
    dataset_size: str = '50m'
    interaction_name: str = 'multi_event'
    default_like_window_seconds: int = 24 * 60 * 60
    lag_seconds: int = 15 * 60

dataset_config = DatasetConfig()

In [9]:
from typing import Literal
from datasets import Dataset, DatasetDict, load_dataset


# YambdaDataset wrapper class (https://huggingface.co/datasets/yandex/yambda)
class YambdaDataset:
    INTERACTIONS = frozenset([
        "likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"
    ])

    def __init__(
        self,
        dataset_type: Literal["flat", "sequential"] = "flat",
        dataset_size: Literal["50m", "500m", "5b"] = "50m"
    ):
        assert dataset_type in {"flat", "sequential"}
        assert dataset_size in {"50m", "500m", "5b"}
        self.dataset_type = dataset_type
        self.dataset_size = dataset_size

    def interaction(self, event_type: Literal[
        "likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"
    ]) -> Dataset:
        assert event_type in YambdaDataset.INTERACTIONS
        return self._download(f"{self.dataset_type}/{self.dataset_size}", event_type)

    def audio_embeddings(self) -> Dataset:
        return self._download("", "embeddings")

    def album_item_mapping(self) -> Dataset:
        return self._download("", "album_item_mapping")

    def artist_item_mapping(self) -> Dataset:
        return self._download("", "artist_item_mapping")

    @staticmethod
    def _download(data_dir: str, file: str) -> Dataset:
        data = load_dataset("yandex/yambda", data_dir=data_dir, data_files=f"{file}.parquet")
        # Returns DatasetDict; extracting the only split
        assert isinstance(data, DatasetDict)
        return data["train"]


In [10]:
yambda_dataset = YambdaDataset(
    dataset_type=dataset_config.dataset_type,
    dataset_size=dataset_config.dataset_size
)
albums = yambda_dataset.album_item_mapping().to_polars()
artists = yambda_dataset.artist_item_mapping().to_polars()

README.md: 0.00B [00:00, ?B/s]

album_item_mapping.parquet:   0%|          | 0.00/54.5M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

artist_item_mapping.parquet:   0%|          | 0.00/38.3M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [11]:
albums

album_id,item_id
u32,u32
1,1491131
1,5109849
1,6735246
2,2859065
3,1859377
…,…
3367688,7302579
3367688,8345353
3367689,269381


In [12]:
artists

artist_id,item_id
u32,u32
1,953587
1,1921481
1,2659068
1,2740764
1,6767564
…,…
1293390,8177055
1293391,8269848
1293392,5930302


In [13]:
listens

uid,timestamp,item_id,played_ratio_pct,track_length_seconds,is_like,is_full_play,is_skip,user_lag_listen_cnt,user_lag_like_cnt,user_lag_full_play_cnt,user_lag_skip_cnt,item_lag_listen_cnt,item_lag_like_cnt,item_lag_full_play_cnt,item_lag_skip_cnt,ui_lag_listen_cnt,ui_lag_like_cnt,ui_lag_full_play_cnt,ui_lag_skip_cnt,user_lag_avg_played_ratio,item_lag_avg_played_ratio,ui_lag_avg_played_ratio
u32,u32,u32,u16,u32,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
100,40110,732449,100,240,false,true,false,0.0,0.0,0.0,0.0,2.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,50.5,0.0
100,40360,3397170,46,130,false,false,true,2.0,0.0,2.0,0.0,4.0,0.0,3.0,1.0,0.0,0.0,0.0,0.0,100.0,83.25,0.0
100,40380,7849270,100,205,false,true,false,2.0,0.0,2.0,0.0,7.0,0.0,4.0,3.0,0.0,0.0,0.0,0.0,100.0,57.428571,0.0
100,41130,6474571,100,245,false,true,false,4.0,0.0,4.0,0.0,2.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,100.0,7.0,0.0
100,41545,7847600,27,250,false,false,true,7.0,0.0,6.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,92.285714,100.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1000000,25277150,7479030,99,155,false,true,false,361.0,3.0,330.0,27.0,786.0,1.0,710.0,60.0,0.0,0.0,0.0,0.0,92.725762,92.410941,0.0
1000000,25277170,9295648,15,120,false,false,true,361.0,3.0,330.0,27.0,4686.0,4.0,3789.0,745.0,0.0,0.0,0.0,0.0,92.725762,84.593257,0.0
1000000,25960045,1914019,100,185,false,true,false,365.0,3.0,332.0,29.0,2505.0,47.0,1488.0,903.0,0.0,0.0,0.0,0.0,92.309589,65.346108,0.0


## Adding multivalent features

Earlier, we loaded the `albums` and `artists` data. Now, based on them, you need to add two new columns with multivalent categorical features to the table:
- `artist_ids` — the list of artists corresponding to the given track;
- `album_ids` — the list of albums corresponding to the given track.

Thus, for each track, these columns should store not a single identifier, but a list of identifiers.

So the logic is as follows

artists_agg:  item_id -> `[artist_id1, artist_id2, ...]`

albums_agg:   item_id -> `[album_id1, album_id2, ...]`

and then

listens + item_id -> artist_ids, album_ids

In [14]:
def join_item_artist_album(
    listens: pl.DataFrame,
    artists: pl.DataFrame,
    albums: pl.DataFrame,
) -> pl.DataFrame:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    artists_agg = artists.group_by("item_id").agg(
    pl.col("artist_id").alias("artist_ids")
    )

    albums_agg = albums.group_by("item_id").agg(
        pl.col("album_id").alias("album_ids")
    )

    listens = (
        listens
        .join(artists_agg, on="item_id", how="left")
        .join(albums_agg, on="item_id", how="left")
    )

    return listens

listens = join_item_artist_album(listens, artists, albums)
tests.test_join_item_artist_album(listens)
del artists, albums
gc.collect()

All good! :)


0

In [15]:
listens.head(4)

uid,timestamp,item_id,played_ratio_pct,track_length_seconds,is_like,is_full_play,is_skip,user_lag_listen_cnt,user_lag_like_cnt,user_lag_full_play_cnt,user_lag_skip_cnt,item_lag_listen_cnt,item_lag_like_cnt,item_lag_full_play_cnt,item_lag_skip_cnt,ui_lag_listen_cnt,ui_lag_like_cnt,ui_lag_full_play_cnt,ui_lag_skip_cnt,user_lag_avg_played_ratio,item_lag_avg_played_ratio,ui_lag_avg_played_ratio,artist_ids,album_ids
u32,u32,u32,u16,u32,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,list[u32],list[u32]
100,40110,732449,100,240,false,true,false,0.0,0.0,0.0,0.0,2.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,50.5,0.0,[94924],[1632076]
100,40360,3397170,46,130,false,false,true,2.0,0.0,2.0,0.0,4.0,0.0,3.0,1.0,0.0,0.0,0.0,0.0,100.0,83.25,0.0,[997338],[403561]
100,40380,7849270,100,205,false,true,false,2.0,0.0,2.0,0.0,7.0,0.0,4.0,3.0,0.0,0.0,0.0,0.0,100.0,57.428571,0.0,[995468],[477338]
100,41130,6474571,100,245,false,true,false,4.0,0.0,4.0,0.0,2.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,100.0,7.0,0.0,[163539],[1412710]


## Splitting the data into training and test sets

Next, you need to write a function that splits the dataset into two parts by time:
- `train` should contain all events except those from the last `test_last_seconds` seconds;
- `test` should contain events from the last `test_last_seconds` seconds, including the boundary.

In this assignment, we will use the last 30 days as the test set.

In [16]:
def temporal_train_test_split(
    df: pl.DataFrame,
    test_last_seconds: float,
    time_column: str = "timestamp",
) -> tuple[pl.DataFrame, pl.DataFrame]:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    max_ts = df.select(pl.col(time_column).max()).item() # to split the data correctly
    split_ts = max_ts - test_last_seconds

    train = df.filter(pl.col(time_column) < split_ts)
    test = df.filter(pl.col(time_column) >= split_ts)

    return train, test

train_listens, test_listens = temporal_train_test_split(listens, test_last_seconds=30 * 24 * 60 * 60)
tests.test_temporal_train_test_split(train_listens, test_listens)
del listens
gc.collect()

All good! :)


60

# 2. Training and evaluating baselines

## Pairwise accuracy

Метрика, которой будет оцениваться качество вашего решения, называется `pairwise accuracy`.

Идея метрики такая: мы хотим проверить, умеет ли модель правильно упорядочивать соседние по времени события одного пользователя. Например, если в паре соседних треков на один был лайк, а на другой нет, то трек с лайком должен получить больший скор.

Функция получает на вход:
- `uids` — идентификаторы пользователей;
- `timestamps` — времена событий;
- `labels` — бинарные таргеты (`0` или `1`);
- `probs` — предсказанные моделью скоры;
- `session_gap_seconds` — максимальный допустимый разрыв по времени между соседними событиями.

Что нужно реализовать:
1. сгруппировать события по пользователям;
2. внутри каждого пользователя отсортировать события по времени;
3. рассмотреть только соседние пары;
4. отбросить пары, у которых разница по времени больше `session_gap_seconds`;
5. оставить только пары, у которых различаются `label`;
6. проверить, совпадает ли порядок по `probs` с порядком по `label`.

Вклад одной пары:
- 1, если модель ранжирует пару правильно;
- 0, если неправильно;
- 0.5, если скоры равны.

Формально, для соседней валидной пары $(i, i+1)$:

$$
m(i, i+1)=
\begin{cases}
1, & \text{если } (y_{i+1}-y_i)(s_{i+1}-s_i) > 0, \\
0.5, & \text{если } s_{i+1} = s_i, \\
0, & \text{иначе.}
\end{cases}
$$

Итоговая метрика:

$$
\text{PairwiseAccuracy} =
\frac{1}{|P|}\sum_{(i,i+1)\in P} m(i,i+1),
$$

где $P$ — множество всех валидных соседних пар по всем пользователям.

Почему мы смотрим именно на соседние пары: в рекомендательных задачах контекст пользователя быстро меняется. Если сравнивать два события, между которыми прошло много времени, такое сравнение уже может быть некорректным: пользователь был в другом состоянии, слушал другую музыку и вообще решал другую задачу. Поэтому мы сравниваем только локальные, соседние по времени события.

По этой же причине здесь не очень подходит ROC AUC. Эта метрика фактически сравнивает позитивные и негативные объекты между собой глобально, в том числе из совершенно разных моментов времени. В результате модель может получить хороший ROC AUC, даже если она плохо упорядочивает близкие по времени события внутри реальной пользовательской последовательности. `Pairwise accuracy` лучше отражает именно качество локального ранжирования, которое важно в этой задаче.

Дополнительно важно, что временной порог в метрике согласован с лагом, который использовался при построении признаков. В признаках мы используем задержку 15 минут, чтобы модель не могла опираться на слишком свежий фидбек, который в реальной системе ещё не успел бы попасть в фичи. В метрике используется тот же порог: если два соседних события находятся друг от друга дальше, чем на 15 минут, мы их не сравниваем. Это делает постановку более согласованной: и при построении признаков, и при оценке качества мы работаем в одном и том же локальном временном окне.

In [ ]:
import numpy as np


def compute_pairwise_accuracy(
    uids: np.array,
    timestamps: np.array,
    labels: np.array,
    probs: np.array,
    session_gap_seconds: int = 15 * 60,
) -> dict[str, float]:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass

tests.test_pairwise_accuracy(compute_pairwise_accuracy)

## Считаем метрики случайного бейзлайна

Перед обучением модели полезно посчитать качество простого случайного бейзлайна. Для этого сгенерируйте для каждого объекта в тестовой выборке случайный скор и посчитайте по нему обе метрики `pairwise accuracy`:
- по таргету `is_like`;
- по таргету `is_full_play`.

На этом шаге нужно:
1. взять тестовую выборку;
2. сгенерировать для каждого объекта случайное значение score;
3. вычислить `pair_accuracy_like`;
4. вычислить `pair_accuracy_full_play`.

В результате у вас должен получиться словарь `metrics` с двумя значениями метрик для случайного ранжирования.

In [ ]:
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

metrics = {
    "pair_accuracy_like": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    ),
    "pair_accuracy_full_play": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    )
}
tests.test_random_baseline_metrics(metrics)

## Считаем метрики popularity baseline

Ещё один простой бейзлайн — ранжировать треки по их популярности в обучающей выборке. Для этого посчитайте, сколько раз каждый `item_id` встретился в `train`, и используйте это значение как score на `test`.

На этом шаге нужно:
1. по обучающей выборке посчитать популярность каждого трека (сколько раз он встретился);
2. присоединить эту популярность к объектам тестовой выборки;
3. для треков, которых не было в `train`, подставить `0`;
4. вычислить `pair_accuracy_like`;
5. вычислить `pair_accuracy_full_play`.

В результате у вас должен получиться словарь `metrics` с двумя значениями метрик для popularity baseline.

In [ ]:
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

metrics = {
    "pair_accuracy_like": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    ),
    "pair_accuracy_full_play": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    )
}
tests.test_popularity_baseline_metrics(metrics)

## Обучаем CatBoost baseline

Теперь обучим ещё один бейзлайн — модель CatBoost. В отличие от предыдущих простых бейзлайнов, здесь модель уже будет использовать признаки из датасета и учиться предсказывать таргет по обучающей выборке.

На этом шаге нужно:
1. взять вещественные признаки `DENSE_COLUMNS` и категориальные признаки `SPARSE_COLUMNS`;
2. обучить `CatBoostClassifier` на задаче предсказания `is_full_play`;
3. получить предсказанные вероятности на тестовой выборке;
4. использовать вероятность полного прослушивания как score;
5. посчитать `pair_accuracy_like`;
6. посчитать `pair_accuracy_full_play`.

Обратите внимание, что модель обучается только на таргете `is_full_play`, но качество нужно измерить по обоим таргетам. Это позволит понять, насколько предсказание лайка переносится на задачу ранжирования по полному прослушиванию.

В результате у вас должен получиться словарь `metrics` с двумя значениями:
- `pair_accuracy_like`;
- `pair_accuracy_full_play`.

In [ ]:
DENSE_COLUMNS: tuple[str, ...] = (
    "user_lag_listen_cnt",
    "user_lag_like_cnt",
    "user_lag_full_play_cnt",
    "user_lag_skip_cnt",
    "item_lag_listen_cnt",
    "item_lag_like_cnt",
    "item_lag_full_play_cnt",
    "item_lag_skip_cnt",
    "ui_lag_listen_cnt",
    "ui_lag_like_cnt",
    "ui_lag_full_play_cnt",
    "ui_lag_skip_cnt",
    "user_lag_avg_played_ratio",
    "item_lag_avg_played_ratio",
    "ui_lag_avg_played_ratio",
)
MULTIVALENT_COLUMNS: tuple[str, ...] = ("artist_ids", "album_ids")
SPARSE_COLUMNS = ("uid", "item_id")
LABEL_COLUMNS = ("is_like", "is_full_play")

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

metrics = {
    "pair_accuracy_like": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    ),
    "pair_accuracy_full_play": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    )
}
tests.test_catboost_baseline_metrics(metrics)

# 3. Concat+MLP модель (3 балла)

В этом задании вам предстоит реализовать первое нейросетевое решение.

Идея модели очень простая:
1. вещественные признаки кодируются с помощью `PiecewiseLinearEncoder`;
2. категориальные и мультивалентные признаки кодируются с помощью `Multisize Unified Embeddings`;
3. все полученные представления конкатенируются в один вектор;
4. этот вектор подаётся в простую полносвязную сеть с `ReLU` в качестве нелинейности.

На этом этапе модель предсказывает только вероятность полного прослушивания, то есть решается одна бинарная задача. Пока что речь о multi-task learning и нескольких головах не идёт.

Таким образом, вам нужно реализовать модель, которая:
- получает dense-, sparse- и multivalent-признаки;
- кодирует каждый тип признаков своим энкодером;
- объединяет все представления;
- пропускает результат через `MLP`;
- выдаёт скалярный score для предсказания полного прослушивания.

## Готовим датасет для обучения

Теперь нужно реализовать класс `RankerDataset`, который будет подготавливать батчи для обучения нейросетевой модели.

В этом задании батчевание происходит **на этапе самого датасета**, а не в `DataLoader`. Здесь это сделано специально: табличные данные уже лежат целиком в памяти, а модель сама по себе небольшая, поэтому узким местом легко становится именно подготовка батчей и перекладывание данных. Если формировать большие батчи на лету, то даже при больших `num_workers` и `prefetch_factor` загрузка данных может стать bottleneck. Поэтому здесь удобнее заранее нарезать датафрейм на батчи и хранить их в готовом виде внутри датасета.

Вам дан каркас класса `RankerDataset`. Нужно реализовать его так, чтобы в конструкторе датафрейм разбивался на батчи размера `batch_size`, а каждый батч преобразовывался в словарь фиксированной структуры.

### Что нужно сделать

1. Проверить, что `batch_size >= 1`.
2. Разбить входной `df` на последовательные батчи размера `batch_size`.
3. Для каждого батча собрать dense-, sparse- и multivalent-признаки, а также таргеты и мета-информацию.
4. Последовательно применить к каждому батчу все функции из `transforms`.
5. Сохранить готовые батчи внутри датасета.
6. Реализовать:
   - `__len__` — количество батчей;
   - `__getitem__` — получение батча по индексу.

### Формат батча

Каждый батч должен быть словарём со следующими ключами:
- `labels`
- `dense_features`
- `sparse_features`
- `multivalent_features`
- `meta`

#### `labels`

Словарь, где ключами являются названия таргетов из `label_columns`, а значениями — одномерные `torch.Tensor` длины `batch_size`.

Пример:

```python
{
    "is_like": ...,
    "is_full_play": ...,
}
```

#### `dense_features`
Тензор размера [batch_size, num_dense_features] типа torch.float32.

#### `sparse_features`
Словарь, где для каждого имени из sparse_columns хранится одномерный тензор длины batch_size.

Пример:

```python
{
    "uid": ...,
    "item_id": ...,
}
```

#### `multivalent_features`
Словарь, где для каждого мультивалентного признака нужно сохранить ещё один словарь с двумя полями:
-  values — один общий плоский тензор со всеми значениями из списков;
- lengths — тензор длин списков для каждого объекта в батче.

Пример:

```python
{
    "artist_ids": {
        "values": ...,
        "lengths": ...,
    },
    "album_ids": {
        "values": ...,
        "lengths": ...,
    },
}
```

Именно в таком формате мультивалентные признаки удобно потом агрегировать внутри модели.

#### `meta`
Словарь с технической информацией о батче:

```python
{
    "timestamp": ...,
    "uid": ...,
    "item_id": ...,
}
```

Что такое transforms

Аргумент transforms — это список функций, которые принимают батч и возвращают преобразованный батч. Эти преобразования нужно применять последовательно к каждому уже собранному батчу.

То есть логика такая:
1. вы собрали батч в нужном формате;
2.	прогнали его через все функции из transforms;
3.	сохранили итоговый батч в датасет.

Это позволит дальше независимо добавлять, например, перенос на устройство, нормализацию, переименование полей или любые другие постобработки.

Дополнительные замечания
- Dense-признаки и таргеты нужно привести к torch.float32.
- Для мультивалентных признаков lengths и values должны быть целочисленными тензорами.
- Последний батч может быть неполным.
- Пустые батчи сохранять не нужно.

В результате ваш RankerDataset должен возвращать готовые батчи, а не отдельные объекты.

In [ ]:
import polars as pl
from typing import Any, Callable

from torch.utils.data import Dataset


class RankerDataset(Dataset):
    def __init__(
        self,
        df: pl.DataFrame,
        transforms: list[Callable[[Any], Any]],
        label_columns: list[str],
        dense_columns: list[str],
        sparse_columns: list[str],
        multivalent_columns: list[str],
        batch_size: int,
    ):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def __len__(self) -> int:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def __getitem__(self, idx: int) -> dict[str, Any]:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

tests.test_ranker_dataset(RankerDataset)

## Multihash transform

Теперь реализуем `MultihashTransform` — преобразование, которое будет переводить sparse- и multivalent-признаки в набор хешей. Этот шаг нужен для дальнейшей работы с unified embeddings.

Идея здесь следующая: вместо того, чтобы хранить отдельную embedding-таблицу под каждый признак или использовать один-единственный хеш, мы отображаем каждый идентификатор сразу в несколько хешей с разными seed. В результате один и тот же исходный ID превращается в небольшой набор индексов. Такой подход предлагается в статье [Unified Embedding: Battle-Tested Feature Representations for Web-Scale ML Systems](https://arxiv.org/pdf/2305.12102) и позволяет компактнее работать с большим количеством категориальных признаков.

Вам дан каркас класса `MultihashTransform`. Нужно реализовать его так, чтобы он применял multihash-преобразование:
- к sparse-признакам;
- к multivalent-признакам.

### Что приходит на вход

На вход `__call__` подаётся `sample` — словарь батча, в котором уже есть поля:
- `sparse_features`;
- `multivalent_features`.

Именно их и нужно преобразовать.

### Что задаётся в конструкторе

- `sparse_features_config` — словарь вида  
  `{feature_name: [seed1, seed2, ...]}`  
  для sparse-признаков;
- `sparse_features_name` — имя поля в `sample`, где лежат sparse-признаки;
- `multivalent_features_config` — словарь такого же вида для multivalent-признаков;
- `multivalent_features_name` — имя поля в `sample`, где лежат multivalent-признаки;
- `cardinality` — размер общего хеш-пространства, то есть все индексы после хеширования должны лежать в диапазоне `[0, cardinality)`.

### Что нужно сделать

Для каждого sparse-признака:
1. взять тензор идентификаторов;
2. для каждого `seed` из конфига отдельно посчитать хеш;
3. привести хеш в диапазон `[0, cardinality)` с помощью взятия остатка;
4. сохранить результат обратно в `sample`.

Для каждого multivalent-признака:
1. взять плоский тензор `values`;
2. для каждого `seed` из конфига отдельно посчитать хеш;
3. привести хеш в диапазон `[0, cardinality)`;
4. сохранить результат обратно в `sample`, не меняя `lengths`.

### Какой формат должен получиться

Если исходный sparse-признак имел форму

```python
[batch_size]
```

то после multihash-преобразования он должен иметь форму

```python
[batch_size, num_hashes]
```

где num_hashes — число seed для этого признака.

Если values у multivalent-признака имели форму

```python
[num_values]
```

то после преобразования должно получиться

```python
[num_values, num_hashes]
```

Поле lengths у multivalent-признаков менять не нужно.

Важные замечания
- Для хеширования нужно использовать mmh3.
- Один и тот же исходный ID при одном и том же seed всегда должен давать один и тот же индекс.
- Разные seed должны давать разные хеш-проекции одного и того же ID.
- Все полученные индексы должны быть целыми числами из диапазона `[0, cardinality)`.
- Преобразование должно происходить in-place: нужно модифицировать sample и вернуть его обратно.

Итоговая идея такая: после MultihashTransform каждый категориальный ID заменяется не одним индексом, а набором индексов, полученных несколькими независимыми хеш-функциями.


In [ ]:
from typing import Any
import mmh3


class MultihashTransform:
    def __init__(
        self,
        sparse_features_config: dict,
        sparse_features_name: str,
        multivalent_features_config: dict,
        multivalent_features_name: str,
        cardinality: int,
    ):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def __call__(self, sample: dict[str, Any]) -> dict[str, Any]:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        return sample


tests.test_multihash_transform(MultihashTransform)

## CategoricalEncoder

Теперь реализуем `CategoricalEncoder` — простой энкодер категориальных признаков. Его задача состоит в том, чтобы по входным индексам возвращать соответствующие embedding-векторы.

В конструктор класса передаётся объект `nn.Embedding`, который нужно сохранить внутри модуля. В методе `forward` на вход подаётся тензор `ids`, а на выходе должен получаться результат применения `embeddings(ids)`.

Иными словами, `CategoricalEncoder` — это тонкая обёртка над `nn.Embedding`.

### Что нужно реализовать

1. Сохранить переданный слой `embeddings`.
2. В `forward` применить его к тензору `ids`.
3. Вернуть получившийся тензор.

### Ожидаемое поведение

Если на вход подан тензор индексов формы

```python
[batch_size]
```

то на выходе должен получиться тензор формы

```python
[batch_size, embedding_dim]
```

Если на вход подан тензор формы

```python
[batch_size, num_hashes]
```

то на выходе должен получиться тензор формы

```python
[batch_size, num_hashes, embedding_dim]
```

То есть энкодер должен работать с произвольной входной формой так же, как обычный nn.Embedding.


In [ ]:
import torch
import torch.nn as nn


class CategoricalEncoder(nn.Module):
    def __init__(self, embeddings: nn.Embedding):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


tests.test_categorical_encoder(CategoricalEncoder)

## MultivalentEncoder

Теперь реализуем `MultivalentEncoder` — энкодер для мультивалентных категориальных признаков. В отличие от обычного категориального признака, здесь каждому объекту соответствует не один ID, а список ID, например список артистов или альбомов.

В конструктор класса передаётся объект `nn.Embedding`, который нужно сохранить внутри модуля. В методе `forward` на вход подаются:
- `ids` — плоский тензор идентификаторов после `MultihashTransform`;
- `lengths` — длины списков для каждого объекта в батче.

Нужно получить embedding для каждого ID, а затем усреднить эмбеддинги внутри каждого списка. Для этого удобно использовать `torch.nn.functional.embedding_bag` с режимом `"mean"`.

### Что нужно реализовать

1. Сохранить переданный слой `embeddings`.
2. В `forward` агрегировать эмбеддинги по спискам с помощью среднего.
3. Вернуть тензор представлений объектов.

### Ожидаемое поведение

Если на вход подан:
- `ids` формы `[num_values, num_hashes]`,
- `lengths` формы `[batch_size]`,

то на выходе должен получиться тензор формы

```python
[batch_size, num_hashes, embedding_dim]
```

То есть для каждого объекта в батче нужно получить по одному усреднённому embedding-представлению для каждого хеша.

In [ ]:
class MultivalentEncoder(nn.Module):
    def __init__(self, embeddings: nn.Embedding):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

tests.test_multivalent_encoder(MultivalentEncoder)

## PiecewiseLinearEncoder

Теперь нужно реализовать `PiecewiseLinearEncoder` — слой для кусочно-линейного кодирования вещественных признаков как в статье [On Embeddings for Numerical Features in Tabular Deep Learning](https://arxiv.org/abs/2203.05556).

Идея этого преобразования такая: каждый вещественный признак разбивается на интервалы по квантилям, а затем значение признака представляется набором кусочно-линейных активаций относительно этих интервалов. В результате один вещественный признак превращается в несколько чисел, которые уже удобнее подавать в табличную нейросетевую модель.

В этом задании вам нужно реализовать:
- `compute_bins`;
- `from_dataset`;
- `__init__`;
- свойство `n_bins`;
- `forward`.

---

### Вход и выход слоя

На вход `forward` подаётся тензор

```python
[batch_size, n_features]
```

где:
- batch_size — число объектов в батче;
- n_features — число вещественных признаков.

На выходе должен получаться тензор

```python
[batch_size, encoded_dim]
```

где encoded_dim — суммарная размерность кусочно-линейного представления всех признаков.

---

Что делает `compute_bins`

Метод `compute_bins(X, n_bins)` должен:
1.	принять тензор X формы `[n_objects, n_features]`;
2.	для каждой колонки отдельно посчитать квантили;
3.	вернуть список тензоров с границами бинов, по одному тензору на каждый признак.

Если бы у всех квантилей были разные значения, то для каждого признака получилось бы ровно n_bins + 1 границ. Но на практике это не всегда так: если в колонке много одинаковых значений, некоторые квантили совпадут. Поэтому после вычисления квантилей нужно удалить повторы через `unique()`.

Именно поэтому реальное число бинов у признака может быть:
- равно n_bins;
- меньше n_bins;
- в крайнем случае равно 1.

---

Что возвращает compute_bins

Метод должен вернуть список длины n_features:

```python
[
    tensor([...]),  # границы бинов для 1-го признака
    tensor([...]),  # границы бинов для 2-го признака
    ...
]
```

Для признака с k бинами тензор границ имеет длину k + 1.

Например:
- если после удаления дублей осталось 5 границ, то это означает 4 бина;
- если осталось 2 границы, то это означает 1 бин.

---

Что делает `from_dataset`

Метод `from_dataset(...)` должен построить готовый объект PiecewiseLinearEncoder по обучающим данным.

Для этого нужно:
1.	взять срез обучающего датафрейма dense_train_df (например, 1_000_000 первых строк);
2.	преобразовать его в тензор;
3.	посчитать биновые границы через compute_bins;
4.	по этим границам построить все внутренние параметры энкодера;
5.	вернуть готовый экземпляр класса.

---

Что такое `weight` и `bias`

Для каждого признака и каждого бина мы заранее строим коэффициенты линейной функции. Если границы очередного бина равны

$$
[a, b],
$$

то для него считаются коэффициенты

$$
w = \frac{1}{b-a}, \qquad c = -\frac{a}{b-a}.
$$

Тогда линейная функция на этом бине имеет вид

$$
w x + c = \frac{x-a}{b-a}.
$$

Именно поэтому всё преобразование потом можно считать векторизованно через

$$
bias + weight \odot x.
$$

Здесь:
- weight хранит наклоны линейных функций;
- bias хранит сдвиги.

---

Почему число бинов может отличаться у разных признаков

После удаления одинаковых квантилей у разных колонок может остаться разное число границ, а значит и разное число бинов.

Например:
- у первого признака может остаться 4 бина;
- у второго — только 2;
- у третьего — только 1.

Чтобы хранить всё в одном тензоре, нужно:
1.	взять максимальное число бинов max_n_bins;
2.	завести weight и bias размера `[n_features, max_n_bins]`;
3.	для признаков с меньшим числом бинов часть позиций оставить пустыми;
4.	затем в forward убрать лишние координаты с помощью mask.

---

Что делает `mask`

`mask` нужен для случая, когда у разных признаков разное число бинов.

Если у всех признаков число бинов одинаковое, то mask можно не использовать и оставить None.

Если число бинов различается, то:
- представление приходится паддить до общего max_n_bins;
- после этого mask указывает, какие координаты нужно оставить, а какие удалить.

Итоговая логика такая:
- сначала строится представление размера [batch_size, n_features, max_n_bins];
- потом оно разворачивается;
- потом лишние паддинговые координаты удаляются через mask.

---

Что делает `single_bin_mask`

single_bin_mask нужен для признаков, у которых после удаления одинаковых квантилей остался ровно один бин.

Это специальный крайний случай. Для таких признаков последняя координата в forward обрабатывается не совсем так же, как для остальных. Поэтому нужен отдельный булевый вектор длины n_features, который отмечает признаки с единственным бином.

Если таких признаков нет, single_bin_mask можно оставить None.

---

Какие крайние случаи нужно учитывать

1. У признака меньше бинов, чем n_bins
Это нормальная ситуация. Она возникает, когда часть квантилей совпала из-за повторяющихся значений. Ничего дополнительно делать не нужно: просто работаем с тем числом бинов, которое реально получилось после `unique()`.

2. У признака ровно один бин
Это тоже допустимая ситуация. Она означает, что после удаления повторов у признака осталось ровно две разные границы.

Такой признак не нужно выбрасывать. Его нужно корректно обработать через single_bin_mask.

3. У признака вообще нет интервала
Если после вычисления квантилей и удаления дублей у признака остаётся только одна уникальная граница, то это означает, что колонка по сути константная и для неё невозможно построить даже один бин.

В этом случае нужно выбросить ошибку:

```python
assert n_bin >= 1, "There is a column with only one unique value"
```

Иными словами:
- 1 бин — допустимо;
- 0 бинов — недопустимо.

4. У всех признаков одинаковое число бинов
В этом случае mask не нужен, и его можно хранить как None.

5. У разных признаков разное число бинов
В этом случае mask обязателен, иначе на выходе останутся лишние координаты от паддинга.

---

Что нужно сделать в `__init__`

В конструкторе нужно:
1.	сохранить список n_bins;
2.	сохранить `weight` и `bias` как buffer;
3.	сохранить mask как buffer, если он не None;
4.	сохранить single_bin_mask как buffer, если он не None.

Здесь именно register_buffer, а не nn.Parameter, потому что это не обучаемые параметры, а заранее посчитанные служебные тензоры.

---

Что должно возвращать свойство `n_bins`

Свойство n_bins должно возвращать список числа бинов для каждого вещественного признака.

Например, если у трёх признаков получилось 4, 2 и 1 бин, то:

```python
encoder.n_bins == [4, 2, 1]
```

---

Что нужно сделать в forward

В forward(x) нужно:
1.	взять вход x формы [batch_size, n_features];
2.	применить векторизованное преобразование через bias и weight;
3.	правильно обрезать значения:
	- первая координата должна быть ограничена сверху;
	- внутренние координаты должны быть ограничены с двух сторон;
	- последняя координата должна обрабатываться отдельно;
4.	учесть специальный случай признаков с одним бином через single_bin_mask;
5.	развернуть последние две размерности;
6.	если есть mask, применить его;
7.	вернуть итоговый тензор признаков.

---

Интуиция про обрезание значений

После вычисления линейных функций получается набор чисел, который нужно превратить в корректное кусочно-линейное представление.

Для этого:
- левая часть должна насыщаться сверху;
- внутренние части должны лежать в диапазоне от 0 до 1;
- правая часть должна насыщаться снизу;
- для признаков с одним бином последняя координата обрабатывается отдельно.

Именно поэтому в forward используются `clamp`, `clamp_min` и `clamp_max`.

---

Подсказка:
- Если не получается решить задачу, то можно обратиться к оригинальной реализации от авторов: https://github.com/yandex-research/rtdl-num-embeddings/tree/a8fc25025c83f2321c63ff127a3bcef83bb1bfb5


In [ ]:
class PiecewiseLinearEncoder(nn.Module):
    @staticmethod
    def compute_bins(
        X: torch.Tensor,
        n_bins: int,
    ) -> list[torch.Tensor]:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    @classmethod
    def from_dataset(cls, dense_train_df, n_bins=32, train_df_slice: int = 1_000_000):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def __init__(self, weight, bias, mask, n_bins, single_bin_mask):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    @property
    def n_bins(self):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


tests.test_piecewise_linear_encoder(PiecewiseLinearEncoder)

## DeepNetwork

Теперь реализуем `DeepNetwork` — обычную полносвязную сеть из линейных слоёв и `ReLU`.

В конструктор передаются:
- `input_dim` — размер входа;
- `hidden_units` — список размеров скрытых слоёв.

Для каждого значения из `hidden_units` нужно добавить `Linear`, а затем `ReLU`. В `forward` нужно просто применить получившуюся сеть к входу `x`.

Если `hidden_units = [h1, h2, ..., hk]`, то для входа формы

```python
[batch_size, input_dim]
```

выход должен иметь форму

```python
[batch_size, hk]
```

In [ ]:
class DeepNetwork(nn.Module):
    def __init__(self, input_dim, hidden_units):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

tests.test_deep_network(DeepNetwork)

## ConcatMLP

Теперь нужно реализовать модель `ConcatMLP` — первую полную нейросетевую модель в этом задании.

Идея модели такая:
- sparse-признаки кодируются через `CategoricalEncoder`;
- multivalent-признаки кодируются через `MultivalentEncoder`;
- вещественные признаки кодируются через `PiecewiseLinearEncoder`;
- все полученные представления конкатенируются в один вектор;
- этот вектор подаётся в `DeepNetwork`;
- поверх выхода `DeepNetwork` применяется линейный слой, который выдаёт итоговый score.

### Что передаётся в конструктор

- `embedding_size` — размер эмбеддингов для категориальных и мультивалентных признаков;
- `deep_units` — размеры скрытых слоёв `DeepNetwork`;
- `input_size` — размер входа в `DeepNetwork` после конкатенации всех представлений;
- `dense_train_df` — обучающие вещественные признаки, по которым нужно построить `PiecewiseLinearEncoder`;
- `n_bins` — число бинов для кусочно-линейного кодирования;
- `train_df_slice` — размер среза датафрейма, на котором строится `PiecewiseLinearEncoder`;
- `cardinality` — размер общего embedding-словаря;
- `output_size` — размер выхода модели.

### Что нужно сделать в `__init__`

1. Создать общий слой `nn.Embedding`.
2. Создать:
   - `CategoricalEncoder`,
   - `MultivalentEncoder`,
   - `PiecewiseLinearEncoder`,
   - `DeepNetwork`,
   - финальный `Linear`.
3. Сохранить все эти модули в поля класса.

### Что приходит в `forward`

На вход подаётся словарь `inputs` с той же структурой, что и в `RankerDataset`:

- `inputs["dense_features"]`
- `inputs["sparse_features"]["item_id"]`
- `inputs["sparse_features"]["uid"]`
- `inputs["multivalent_features"]["artist_ids"]["values"]`
- `inputs["multivalent_features"]["artist_ids"]["lengths"]`
- `inputs["multivalent_features"]["album_ids"]["values"]`
- `inputs["multivalent_features"]["album_ids"]["lengths"]`

### Что нужно сделать в `forward`

1. Закодировать `item_id` и `uid` через `CategoricalEncoder`.
2. Закодировать `artist_ids` и `album_ids` через `MultivalentEncoder`.
3. Закодировать dense-признаки через `PiecewiseLinearEncoder`.
4. Развернуть категориальные и мультивалентные представления по последним размерностям.
5. Сконкатенировать все представления по последней оси.
6. Передать результат в `DeepNetwork`.
7. Применить финальный линейный слой.
8. Вернуть итоговый output.

### Ожидаемое поведение

Если на вход подан батч размера `batch_size`, то выход модели должен иметь форму

```python
[batch_size, output_size]

In [ ]:
class ConcatMLP(nn.Module):
    def __init__(
        self,
        embedding_size,
        deep_units,
        input_size,
        dense_train_df,
        n_bins,
        train_df_slice,
        cardinality=65536,
        output_size=1,
    ):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, inputs: dict) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


## to_device

Теперь нужно реализовать функцию `to_device`, которая рекурсивно переносит все тензоры в произвольной вложенной структуре данных на заданное устройство.

Такая функция понадобится дальше, чтобы удобно переносить целый батч на CPU или GPU, не разбирая вручную каждое поле.

### Что приходит на вход

- `obj` — произвольный объект;
- `device` — устройство, на которое нужно перенести тензоры.

`obj` может быть:
- `torch.Tensor`;
- `dict`;
- `list`;
- `tuple`;
- любым другим объектом.

### Что нужно сделать

- если `obj` — это `torch.Tensor`, нужно вернуть `obj.to(device)`;
- если `obj` — это `dict`, нужно рекурсивно применить `to_device` ко всем значениям;
- если `obj` — это `list` или `tuple`, нужно рекурсивно применить `to_device` ко всем элементам и сохранить исходный тип контейнера;
- для всех остальных объектов нужно просто вернуть их без изменений.

### Зачем это нужно

Батч в этом задании представляет собой вложенный словарь с dense-, sparse-, multivalent-признаками, таргетами и мета-информацией. Чтобы перед обучением перенести такой батч на нужное устройство, удобно иметь одну универсальную функцию, которая умеет делать это рекурсивно.

In [ ]:
def to_device(obj, device: torch.device | str):
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass

## Train Loop

Далее вам нужно написать train loop, в котором будет происходить обучение вашей модели. Добавьте в него подсчёт `pairwise accuracy` на тестовом наборе данных после каждой эпохи.

Также предусмотрите поддержку `output_size=1` и `output_size=2`. Это нужно, потому что далее модель будет решать сразу две задачи:
- предсказание лайка;
- предсказание полного прослушивания.

**В случае `output_size=1` обучаемся только на предсказание вероятности полного прослушивания. В случае `output_size=2` на лайк и полное прослушивание.**

Каждая задача должна иметь одинаковый вклад в общую функцию потерь, поэтому никаких дополнительных перевзвешиваний добавлять не нужно. Используйте `BCEWithLogitsLoss`.

В случае `output_size=1` нужно вернуть:

```python
{
    "pair_accuracy_full_play": ...,
    "pair_accuracy_full_play_like_pairs": ....
}
```

при `output_size=2` нужно вернуть

```python
{
    "pair_accuracy_full_play": ...,
    "pair_accuracy_full_play_like_pairs": ...
    "pair_accuracy_like": ...,
    "pair_accuracy_like_full_play_pairs": ...
}
```

In [ ]:
def train_model(
    model,
    train_loader,
    test_loader,
    epochs,
    lr,
    train_log_every,
):
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass

## Обучение

Вам нужно подобрать параметры обучение так, чтобы пройти тест.

In [ ]:
multihash_transform = MultihashTransform(
    sparse_features_config={
        'item_id': ,
        'uid': ,
    },
    sparse_features_name="sparse_features",
    multivalent_features_config={
        'artist_ids': ,
        'album_ids': ,
    },
    multivalent_features_name="multivalent_features",
    cardinality=,
)

train_dataset = RankerDataset(
    train_listens,
    [multihash_transform],
    label_columns=list(LABEL_COLUMNS),
    dense_columns=list(DENSE_COLUMNS),
    sparse_columns=list(SPARSE_COLUMNS),
    multivalent_columns=list(MULTIVALENT_COLUMNS),
    batch_size=,
)

test_dataset = RankerDataset(
    test_listens,
    [multihash_transform],
    label_columns=list(LABEL_COLUMNS),
    dense_columns=list(DENSE_COLUMNS),
    sparse_columns=list(SPARSE_COLUMNS),
    multivalent_columns=list(MULTIVALENT_COLUMNS),
    batch_size=,
)

train_listens_dense_million = train_listens[DENSE_COLUMNS].slice(0, 1_000_000)
del train_listens
del test_listens
gc.collect()

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda batch: batch[0],
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda batch: batch[0],
)

In [ ]:
model_concat_mlp = ConcatMLP(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
)

metrics = train_model(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
)

tests.test_model_concat_mlp_metrics(metrics)

# 4. DCN-v2 (3 балла)

В этом задании вам нужно реализовать cross-слой из статьи [DCN V2: Improved Deep & Cross Network and Practical Lessons for Web-scale Learning to Rank Systems](https://arxiv.org/pdf/2008.13535) в варианте **Mixture of Low-Rank Experts**.

Кроме того, дополнительно к `DeepNetwork` вам нужно реализовать ещё две архитектуры:
- `ResDeepNetwork`;
- `DenseDeepNetwork`.

После этого вам нужно будет провести небольшой анализ и сравнить качество разных подходов.

## Mixture Low Rank Cross Layer

Вам нужно реализовать `MixtureLowRankCrossLayer` и `MixtureLowRankCrossNetwork` — low-rank вариант cross-слоя из DCN V2 со смесью экспертов.

Идея этого блока такая: входной вектор признаков несколько раз пропускается через специальные cross-слои, которые моделируют явные взаимодействия между признаками. В отличие от обычного полносвязного слоя, здесь новое представление строится через произведение исходного входа `x0` и преобразования текущего состояния `xl`.

В этой реализации каждый cross-слой состоит из смеси low-rank экспертов:
- каждый эксперт задаёт своё low-rank преобразование;
- затем gate-сеть вычисляет веса экспертов;
- итоговое обновление получается как взвешенная сумма выходов всех экспертов.

### Что нужно реализовать в `MixtureLowRankCrossLayer`

В конструкторе нужно:
1. сохранить `input_dim`, `num_experts` и `rank`;
2. создать обучаемые параметры:
   - `U` формы `[num_experts, input_dim, rank]`,
   - `V` формы `[num_experts, input_dim, rank]`,
   - `bias` формы `[input_dim]`;
3. создать `gate` — линейный слой, который по `xl` предсказывает веса экспертов;
4. инициализировать параметры.

В `forward(x0, xl)` нужно:
1. применить low-rank преобразование к `xl` через `V`, а затем через `U`;
2. прибавить `bias`;
3. умножить результат поэлементно на `x0`;
4. посчитать веса экспертов через `softmax(gate(xl))`;
5. смешать выходы экспертов с этими весами;
6. прибавить residual `xl`.

Итоговый выход должен иметь ту же форму, что и вход:
```python
[batch_size, input_dim]
```

Что нужно реализовать в `MixtureLowRankCrossNetwork`

Это просто стек из нескольких MixtureLowRankCrossLayer.

В конструкторе нужно:
1. создать num_layers cross-слоёв;
2. сохранить их в nn.ModuleList.

В forward(x) нужно:
1. сохранить исходный вход как x0;
2. завести текущее состояние xl = x;
3. последовательно прогнать xl через все cross-слои;
4. вернуть результат последнего слоя.

Важная деталь

Во всей сети x0 остаётся фиксированным и всегда равен исходному входу, а xl меняется от слоя к слою.

Ожидаемое поведение
- MixtureLowRankCrossLayer принимает x0 и xl формы [batch_size, input_dim] и возвращает тензор той же формы.
- MixtureLowRankCrossNetwork принимает x формы [batch_size, input_dim] и возвращает тензор той же формы.



In [ ]:
class MixtureLowRankCrossLayer(nn.Module):
    def __init__(self, input_dim: int, num_experts: int, rank: int):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, x0: torch.Tensor, xl: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


class MixtureLowRankCrossNetwork(nn.Module):
    def __init__(self, input_dim: int, num_layers: int, num_experts: int, rank: int):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


tests.test_mixture_low_rank_cross_network(MixtureLowRankCrossLayer, MixtureLowRankCrossNetwork)

## ResDeepNetwork

Теперь вам нужно реализовать `ResidualMLPBlock` и `ResDeepNetwork` — residual-вариант глубокой полносвязной сети.

Идея здесь такая: вместо обычной последовательности `Linear -> ReLU -> Linear -> ReLU` каждый блок дополнительно использует skip connection. Это помогает стабилизировать обучение и облегчает прохождение градиентов через глубокую сеть.

### ResidualMLPBlock

`ResidualMLPBlock` — это residual-блок из двух линейных слоёв.

В конструкторе нужно:
- создать первый линейный слой;
- создать второй линейный слой;
- при необходимости добавить проекцию `proj`, если размер входа `in_dim` не совпадает с размером выхода `out_dim`.

В `forward` нужно:
1. пропустить вход через MLP-ветку;
2. прибавить skip connection;
3. если размеры не совпадают, сначала применить `proj` к входу;
4. после сложения применить `ReLU`.

Если `in_dim == out_dim`, то в skip connection можно использовать вход `x` напрямую. Если размеры отличаются, нужно сначала перевести вход в размерность `out_dim` через линейную проекцию.

### ResDeepNetwork

`ResDeepNetwork` — это последовательность residual-блоков.

В конструкторе передаются:
- `input_dim` — размер входа;
- `hidden_units` — список выходных размерностей residual-блоков.

Нужно построить цепочку блоков так, чтобы каждый следующий блок принимал выход предыдущего. Для этого можно завести размеры

```python
[input_dim] + hidden_units
```

и затем создать блоки между соседними размерностями.

В `forward` нужно просто последовательно прогнать вход через все блоки.

Ожидаемое поведение

Если `hidden_units = [h1, h2, ..., hk]`, то для входа формы

```python
[batch_size, input_dim]
```

выход ResDeepNetwork должен иметь форму

```python
[batch_size, hk]
```

   

In [ ]:
class ResidualMLPBlock(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


class ResDeepNetwork(nn.Module):
    def __init__(self, input_dim: int, hidden_units: list[int]):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


tests.test_res_deep_network(ResidualMLPBlock, ResDeepNetwork)

## DenseDeepNetwork

Теперь вам нужно реализовать `DenseDeepNetwork` — полносвязную сеть с dense connectivity.

Идея этой архитектуры такая: каждый следующий слой получает на вход не только выход предыдущего слоя, но и все предыдущие представления, включая исходный вход. То есть на каждом шаге вход в слой строится как конкатенация

```python
x, h_1, h_2, ..., h_{i-1}
```

Это позволяет каждому слою напрямую использовать более ранние признаки и уже посчитанные представления.

При этом важно: конкатенация используется только для формирования входа в следующий слой. Выход самого очередного слоя — это обычный тензор размера, заданного в hidden_units, а не накопленная конкатенация всех прошлых выходов.

Что нужно сделать

В конструкторе передаются:
- input_dim — размер исходного входа;
- hidden_units — список размерностей скрытых слоёв.

Нужно:
1. создать последовательность линейных слоёв;
2. учесть, что размер входа в каждый следующий слой растёт, потому что к нему конкатенируются все предыдущие выходы.

В forward нужно:
1. завести список features, который сначала содержит только исходный вход x;
2. на каждом шаге сконкатенировать все тензоры из features;
3. применить очередной линейный слой и ReLU;
4. добавить новый выход в features;
5. вернуть выход последнего слоя.

Ожидаемое поведение

Если `hidden_units = [h1, h2, ..., hk]`, то для входа формы

```python
[batch_size, input_dim]
```

выход должен иметь форму

```pythonj
[batch_size, hk]
```

In [ ]:
class DenseDeepNetwork(nn.Module):
    def __init__(self, input_dim: int, hidden_units: list[int]):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


tests.test_dense_deep_network(DenseDeepNetwork)

## DCN V2

Далее вам нужно реализовать итогвую `DCN-v2` модель. Все аналогично `ConcatMLP` модели, но добавляется новый cross-слой, а так же возможность выбрать deep часть. Испольуйте `build_deep_network`.

In [ ]:
def build_deep_network(
    input_dim: int,
    hidden_units: list[int],
    deep_type: str = "mlp",
) -> nn.Module:
    """Factory for the deep tower: ``mlp``, ``resnet``, or ``densenet``."""
    if deep_type == "mlp":
        return DeepNetwork(input_dim, hidden_units)
    if deep_type == "resnet":
        return ResDeepNetwork(input_dim, hidden_units)
    if deep_type == "densenet":
        return DenseDeepNetwork(input_dim, hidden_units)
    raise ValueError(f"Unknown deep_type={deep_type!r}, expected 'mlp', 'resnet', or 'densenet'")


In [ ]:
class DCNV2(nn.Module):
    def __init__(
        self,
        embedding_size,
        cross_layers,
        deep_units,
        input_size,
        dense_train_df,
        n_bins,
        train_df_slice,
        cardinality=65536,
        num_experts: int = 4,
        low_rank: int = 32,
        deep_network: str = "mlp",
        output_size: int = 2,
    ):
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


    def forward(self, inputs: dict) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass


tests.test_dcnv2(DCNV2)

## Обучение mlp, resnet, densenet с cross-слоями

Ваша задача подобрать параметры для обучения `DCNV2` с `mlp` частью так, чтобы был пройден тест. А так же сравнить этот варинат с `densenet` и `resnet`.

In [ ]:
model_dcnv2_mlp = DCNV2(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
    deep_network="mlp",
)

metrics = train_model(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
)

tests.test_model_dcnv2_mlp_metrics(metrics)

In [ ]:
model_dcnv2_resnet = DCNV2(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
    deep_network="resnet",
)

metrics = train_model(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
)

In [ ]:
model_dcnv2_resnet = DCNV2(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
    deep_network="densenet",
)

metrics = train_model(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
)

# 5. Многоголовость (2 балла)

Одно из важных преимуществ нейросетей состоит в том, что их можно обучать сразу на несколько таргетов. Например, модель может одновременно решать задачу регрессии (предсказывать число секунд прослушивания) и несколько задач классификации (предсказывать лайк, полное прослушивание и другие типы пользовательского фидбека). На практике таких таргетов может быть довольно много, вплоть до десятков (https://blog.reachsumit.com/posts/2023/04/the-twitter-ml-algo/).

Преимущество такого подхода в том, что модель сразу учится строить общее представление объекта, полезное для разных продуктовых сигналов. После этого уже на этапе анализа или A/B-эксперимента можно подбирать веса разных голов и собирать итоговый скор так, чтобы он лучше соответствовал целям продукта.

В этом задании вы обучите двухголовую нейросетевую модель, которая будет одновременно предсказывать:
- лайк;
- полное прослушивание.

После этого вам нужно будет построить парето-фронт и подобрать такие веса для агрегированного score, чтобы он хорошо работал сразу в двух задачах:
- ранжирование по лайкам;
- ранжирование по полным прослушиваниям.

Если удаётся подобрать хорошие веса, это означает, что итоговое ранжирование одновременно хорошо оптимизирует оба сигнала: пользователи и чаще ставят лайки, и чаще дослушивают треки до конца. Именно такой компромисс обычно и важен для бизнеса.

В конце соберите все результаты в одну таблицу и напишите вывод.

## Обучение DCNV2 с двумя головами

Вам нужно подобрать параметры обучения так, чтобы пройти тест. В этом задании `output_size=2`, `deep_network='mlp'`.

In [ ]:
model_multitask_dcnv2_mlp = DCNV2(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
    deep_network="mlp",
    output_size=2,
)

metrics = train_model(
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass
)

tests.test_model_multitask_dcnv2_mlp_metrics(metrics)

## Результаты

| метод | pair_accuracy_like | pair_accuracy_full_play |
|-------|-------------------:|------------------------:|
| random |  |  |
| popular |  |  |
| catboost |  |  |
| concatmlp | |  |
| dcnv2_mlp |  |  |
| dcnv2_resnet |  |  |
| dcnv2_densenet |  |  |
| model_multitask_dcnv2_mlp |  |  |

Вывод:

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################



## Строим парето-фронт

В этом задании вам нужно построить парето-фронт для двухголовой модели.

В нашем случае парето-фронт — это множество компромиссов между двумя целями:
- качеством ранжирования лайков;
- качеством ранжирования полных прослушиваний.

Идея такая: у модели есть две головы, одна предсказывает вероятность лайка, а другая — вероятность полного прослушивания. Из этих двух предсказаний можно собрать итоговый score как взвешенную сумму

$$
s = \alpha \cdot P(\text{like}) + (1 - \alpha) \cdot P(\text{full\_play}),
$$

где $\alpha \in [0, 1]$.

Важно, что в реальной системе ранжирование всё равно обычно происходит **по одному итоговому score**. Даже если модель предсказывает сразу несколько сигналов, на этапе выдачи объектов нужно отсортировать их по одному числу. Поэтому после обучения многоголовой модели нужно понять, как именно агрегировать выходы разных голов в единый score. Как раз для этого и строится парето-фронт.

При разных значениях $\alpha$ мы получаем разные итоговые ранжирования, а значит и разные значения метрик:
- `pair_accuracy_like`;
- `pair_accuracy_full_play`.

От вас требуется:
1. перебрать разные значения $\alpha$;
2. для каждого значения посчитать итоговый score;
3. вычислить `pair_accuracy_like` и `pair_accuracy_full_play`;
4. построить scatter plot:
   - по оси OX — `pair_accuracy_like`;
   - по оси OY — `pair_accuracy_full_play`.

После этого посмотрите на получившийся график и напишите, какие выводы можно сделать. Используйте хотя бы 100 точек. Возьмите модель из предыдущего пункта.

In [ ]:
import matplotlib.pyplot as plt

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

Выводы по графику:

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

## Подберите итоговый score

Теперь вам нужно подобрать такое значение $\alpha$, при котором итоговый score удовлетворяет ограничениям в тесте.

Напомним, что итоговый score строится как

$$
s = \alpha \cdot P(\text{like}) + (1 - \alpha) \cdot P(\text{full\_play}).
$$

На это можно смотреть как на выбор итогового продуктового компромисса: насколько система готова разменивать качество по лайкам на качество по полным прослушиваниям. В реальной задаче именно такой выбор и приходится делать при построении финального ранжирования.

В этом задании вам нужно:
1. подобрать значение $\alpha$;
2. посчитать итоговый score с этим весом;
3. убедиться, что он проходит ограничения из теста.

In [ ]:
alpha =

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

metrics = {
    "pair_accuracy_like": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    ),
    "pair_accuracy_full_play": compute_pairwise_accuracy(
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
    ),
}

tests.test_model_multitask_dcnv2_mlp_combined_score_metrics(metrics)